# 01 — SEC Filings Downloader

**Job:** CIK → SEC submissions API → 13F-HR filings → *content-verified* information-table XML → `data/filings.parquet` + cached XML.

**Lessons baked into `src/sec.py` (from earlier failed attempts):**
1. **403 errors** — the SEC requires a descriptive `User-Agent` with contact info. Set yours in `src/config.py`.
2. **Never hardcode a `Host` header** — the old code sent `Host: data.sec.gov` to `www.sec.gov`, silently breaking Archives requests.
3. **`primary_doc.xml` is the cover page, not the holdings** — the old keyword search picked it first, which is why holdings came back empty. We now *download and verify* that a candidate XML actually contains `<infoTable>` rows before accepting it.
4. **Quarter = `reportDate`, not `filingDate`** — 13Fs are filed ~45 days after quarter end, so filing-date quarters are shifted by one.
5. **Amendments (13F-HR/A) supersede originals** — we keep the latest filing per report period.
6. **Rate limiting + retries** — SEC allows ≤10 req/s; we sleep between requests and retry 403/429/5xx with backoff instead of hiding errors in a bare `except`.

In [1]:
# --- setup (run this cell first, every session) ---
import os, sys, pathlib

# Option A: Google Drive (data persists across sessions -- recommended)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: uploaded zip directly to Colab (data lost on disconnect)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# Option C: running locally
PROJECT_ROOT = pathlib.Path(__file__).resolve().parents[1] if "__file__" in dir() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

# Install dependencies if needed (uncomment on Colab)
# import subprocess; subprocess.run(["pip", "install", "-q", "pyarrow", "requests", "pandas", "matplotlib", "numpy"])

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Project root:", PROJECT_ROOT)
print("Ready.")

Project root: /content/13F-Analytics
Ready.


In [2]:
from src import config
from src.utils import make_session
from src.sec import build_filings_dataset

print("Manager:", config.MANAGER_NAME, "| CIK:", config.CIK)
print("User-Agent:", config.USER_AGENT)
assert "your_email" not in config.USER_AGENT, (
    "Edit USER_AGENT in src/config.py with your real name/email first — "
    "the SEC rejects anonymous requests with 403."
)

Manager: Berkshire Hathaway | CIK: 0001067983
User-Agent: Karanveer Singh klnu5@asu.edu


In [3]:
session = make_session()
filings = build_filings_dataset(session, n_quarters=config.N_QUARTERS)
filings

,accessionNumber,form,filingDate,reportDate,quarter,primaryDocument,xml_file,xml_url,local_xml_path
0,0001193125-26-226661,13F-HR,2026-05-15,2026-03-31,2026Q1,xslForm13F_X02/primary_doc.xml,53405.xml,https://www.sec.gov/Archives/edgar/data/106798...,/content/13F-Analytics/data/raw_xml/2026Q1_000...
1,0001193125-26-054580,13F-HR,2026-02-17,2025-12-31,2025Q4,xslForm13F_X02/primary_doc.xml,50240.xml,https://www.sec.gov/Archives/edgar/data/106798...,/content/13F-Analytics/data/raw_xml/2025Q4_000...
2,0001193125-25-282901,13F-HR,2025-11-14,2025-09-30,2025Q3,xslForm13F_X02/primary_doc.xml,46994.xml,https://www.sec.gov/Archives/edgar/data/106798...,/content/13F-Analytics/data/raw_xml/2025Q3_000...
3,0000950123-25-008343,13F-HR,2025-08-14,2025-06-30,2025Q2,xslForm13F_X02/primary_doc.xml,43977.xml,https://www.sec.gov/Archives/edgar/data/106798...,/content/13F-Analytics/data/raw_xml/2025Q2_000...
4,0000950123-25-008361,13F-HR/A,2025-08-14,2025-03-31,2025Q1,xslForm13F_X02/primary_doc.xml,43981.xml,https://www.sec.gov/Archives/edgar/data/106798...,/content/13F-Analytics/data/raw_xml/2025Q1_000...
5,0000950123-25-002701,13F-HR,2025-02-14,2024-12-31,2024Q4,xslForm13F_X02/primary_doc.xml,39042.xml,https://www.sec.gov/Archives/edgar/data/106798...,/content/13F-Analytics/data/raw_xml/2024Q4_000...
6,0000950123-24-011775,13F-HR,2024-11-14,2024-09-30,2024Q3,xslForm13F_X02/primary_doc.xml,36917.xml,https://www.sec.gov/Archives/edgar/data/106798...,/content/13F-Analytics/data/raw_xml/2024Q3_000...
7,0000950123-24-008740,13F-HR,2024-08-14,2024-06-30,2024Q2,xslForm13F_X02/primary_doc.xml,34725.xml,https://www.sec.gov/Archives/edgar/data/106798...,/content/13F-Analytics/data/raw_xml/2024Q2_000...


In [4]:
# Sanity checks: every quarter should have a verified holdings XML
assert filings["local_xml_path"].notna().all(), "Some filings have no verified infoTable XML"
assert filings["quarter"].is_unique
filings[["quarter", "form", "filingDate", "reportDate", "xml_file"]]

,quarter,form,filingDate,reportDate,xml_file
0,2026Q1,13F-HR,2026-05-15,2026-03-31,53405.xml
1,2025Q4,13F-HR,2026-02-17,2025-12-31,50240.xml
2,2025Q3,13F-HR,2025-11-14,2025-09-30,46994.xml
3,2025Q2,13F-HR,2025-08-14,2025-06-30,43977.xml
4,2025Q1,13F-HR/A,2025-08-14,2025-03-31,43981.xml
5,2024Q4,13F-HR,2025-02-14,2024-12-31,39042.xml
6,2024Q3,13F-HR,2024-11-14,2024-09-30,36917.xml
7,2024Q2,13F-HR,2024-08-14,2024-06-30,34725.xml


**Output:** `data/processed/filings.parquet` + one cached XML per quarter in `data/raw_xml/`. Next: `02_download_xml.ipynb`.